# CodeT5 Retraining with Class-Weighted Loss

**Identical to `04_1_train_full.ipynb` except for one change:**  
The cross-entropy loss is now weighted by inverse class frequency so that
minority CWEs (e.g. CWE188, CWE176, CWE244) are penalised more heavily
when misclassified.

**What changed vs original:**
- Added `compute_class_weights()` after data loading
- `train_one_epoch()` now passes `class_weights` to a manual `CrossEntropyLoss`
  instead of relying on HuggingFace's internal unweighted loss
- Everything else — split, tokenizer, model, optimizer, epochs, batch size — is identical

**Saved to:** `codet5_juliet_weighted/`

In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from tqdm import tqdm

In [2]:
# ── Identical to original ──────────────────────────────────────────────────────
CSV_PATH   = "/Users/rohanbabbar/Documents/Final Year/cwe/Archive/juliet_cwe_dataset.csv"
MODEL_NAME = "Salesforce/codet5-small"
MAX_LENGTH = 256
BATCH_SIZE = 4
EPOCHS     = 2
LR         = 5e-5
SAVE_DIR   = "codet5_juliet_weighted"   # ← different save path so original is preserved
# ──────────────────────────────────────────────────────────────────────────────

device = (
    "mps"  if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"🚀 Using device: {device}")

🚀 Using device: mps


In [3]:
# ── Identical to original ──────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
df = df[["code_clean", "cwe_index"]].dropna()
df["cwe_index"] = df["cwe_index"].astype(int)

print("📦 Dataset loaded")
print(f"Total samples: {len(df)}")
print(df["cwe_index"].value_counts().head())

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["cwe_index"]
)

print(f"\n🧪 Train samples: {len(train_df)}")
print(f"🧪 Val samples:   {len(val_df)}")

📦 Dataset loaded
Total samples: 179839
cwe_index
2      16314
1      14938
110    14844
11     12282
7       9150
Name: count, dtype: int64

🧪 Train samples: 143871
🧪 Val samples:   35968


In [4]:
# ── NEW: compute inverse-frequency class weights ───────────────────────────────
# Formula: w_c = total_samples / (num_classes * samples_in_class_c)
# Minority classes get higher weight → their misclassifications hurt more

num_classes   = int(train_df["cwe_index"].max()) + 1
class_counts  = train_df["cwe_index"].value_counts().sort_index()
total_samples = len(train_df)

# Build weight vector for all class indices 0..num_classes-1
# Classes with no training samples get weight 0 (won't appear in loss)
weights = torch.zeros(num_classes, dtype=torch.float32)
for idx, count in class_counts.items():
    weights[idx] = total_samples / (num_classes * count)

class_weights = weights.to(device)

print(f"🔢 Number of classes : {num_classes}")
print(f"⚖️  Weight range      : {weights[weights>0].min():.4f} – {weights.max():.4f}")
print()

# Show weights for your 16 confused CWEs specifically
# Load CWE name mapping for display
mapping_df = pd.read_csv(CSV_PATH)[["cwe_index", "cwe"]].drop_duplicates()
cwe_map    = dict(zip(mapping_df["cwe_index"], mapping_df["cwe"]))

confused_indices = [
    i for i, cwe in cwe_map.items()
    if cwe in {
        "CWE188","CWE176","CWE121","CWE122","CWE123","CWE124",
        "CWE126","CWE127","CWE134","CWE15", "CWE190","CWE244","CWE114"
    }
]

print("Weights for the 13 confused CWEs (higher = model penalised more for errors):")
for idx in sorted(confused_indices):
    count = class_counts.get(idx, 0)
    print(f"  {cwe_map[idx]:<10} (index {idx:>3})  "
          f"train_samples={count:>6}  weight={weights[idx]:.4f}")

🔢 Number of classes : 118
⚖️  Weight range      : 0.0934 – 203.2076

Weights for the 13 confused CWEs (higher = model penalised more for errors):
  CWE114     (index   0)  train_samples=  1424  weight=0.8562
  CWE121     (index   1)  train_samples= 11950  weight=0.1020
  CWE122     (index   2)  train_samples= 13051  weight=0.0934
  CWE123     (index   3)  train_samples=   370  weight=3.2953
  CWE124     (index   4)  train_samples=  4763  weight=0.2560
  CWE126     (index   5)  train_samples=  3325  weight=0.3667
  CWE127     (index   6)  train_samples=  4763  weight=0.2560
  CWE134     (index   7)  train_samples=  7320  weight=0.1666
  CWE15      (index   8)  train_samples=   122  weight=9.9938
  CWE176     (index   9)  train_samples=   125  weight=9.7540
  CWE188     (index  10)  train_samples=    62  weight=19.6653
  CWE190     (index  11)  train_samples=  9825  weight=0.1241
  CWE244     (index  22)  train_samples=   118  weight=10.3326


In [5]:
# ── Identical to original ──────────────────────────────────────────────────────
class JulietDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        code  = str(self.texts[idx])
        label = int(self.labels[idx])
        encoding = self.tokenizer(
            code,
            truncation=True,
            padding="max_length",
            max_length=self.max_len
        )
        item = {k: torch.tensor(v) for k, v in encoding.items()}
        item["labels"] = torch.tensor(label)
        return item


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
print("\n✅ Tokenizer loaded")

train_dataset = JulietDataset(train_df["code_clean"], train_df["cwe_index"], tokenizer, MAX_LENGTH)
val_dataset   = JulietDataset(val_df["code_clean"],   val_df["cwe_index"],   tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print("\n✅ Datasets & dataloaders ready")

/Users/rohanbabbar/miniconda3/envs/cwe/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



✅ Tokenizer loaded

✅ Datasets & dataloaders ready


In [6]:
# ── Identical to original ──────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=LR)
print("✅ Model & optimizer ready")

Some weights of T5ForSequenceClassification were not initialized from the model checkpoint at Salesforce/codet5-small and are newly initialized: ['classification_head.dense.bias', 'classification_head.dense.weight', 'classification_head.out_proj.bias', 'classification_head.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model & optimizer ready


In [7]:
# ── CHANGED: train_one_epoch now uses weighted CrossEntropyLoss ────────────────
# The only difference from the original is these two lines inside the loop:
#   loss_fn = nn.CrossEntropyLoss(weight=class_weights)
#   loss    = loss_fn(outputs.logits, labels)
# instead of using outputs.loss (which is unweighted internally by HuggingFace)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)  # created once, reused every batch


def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Training"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()

        # Pass labels=None so HuggingFace does NOT compute its own loss
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # Compute weighted loss manually
        loss = loss_fn(outputs.logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


# ── Identical to original ──────────────────────────────────────────────────────
def eval_model(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            total_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    return total_loss / len(dataloader), all_labels, all_preds


print("✅ Training & evaluation functions ready")

✅ Training & evaluation functions ready


In [8]:
# ── Identical to original ──────────────────────────────────────────────────────
for epoch in range(EPOCHS):
    print(f"\n================= EPOCH {epoch+1}/{EPOCHS} =================")

    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    print(f"📉 Train loss: {train_loss:.4f}")

    val_loss, y_true, y_pred = eval_model(model, val_loader, device)
    print(f"📉 Val loss:   {val_loss:.4f}")

    print("\n📊 Classification Report (Validation):")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    print("📊 Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

print("\n✅ Training complete!")

os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"\n💾 Model & tokenizer saved to: {SAVE_DIR}")


================= EPOCH 1/2 =================


Training: 100%|██████████| 35968/35968 [2:44:36<00:00,  3.64it/s]  


📉 Train loss: 0.2138


Evaluating: 100%|██████████| 8992/8992 [10:38<00:00, 14.08it/s]


📉 Val loss:   0.0056

📊 Classification Report (Validation):
              precision    recall  f1-score   support

           0     0.9753    1.0000    0.9875       356
           1     1.0000    0.9993    0.9997      2988
           2     1.0000    0.9991    0.9995      3263
           3     1.0000    1.0000    1.0000        92
           4     1.0000    0.9975    0.9987      1191
           5     1.0000    1.0000    1.0000       831
           6     1.0000    1.0000    1.0000      1191
           7     1.0000    0.9951    0.9975      1830
           8     1.0000    1.0000    1.0000        30
           9     1.0000    1.0000    1.0000        31
          10     1.0000    1.0000    1.0000        16
          11     1.0000    0.9980    0.9990      2457
          12     1.0000    0.9983    0.9992      1816
          13     1.0000    0.9902    0.9951       713
          14     0.9519    1.0000    0.9754       713
          15     1.0000    1.0000    1.0000         8
          16     1.00

Training: 100%|██████████| 35968/35968 [2:38:23<00:00,  3.78it/s]  


📉 Train loss: 0.0071


Evaluating: 100%|██████████| 8992/8992 [10:55<00:00, 13.72it/s]


📉 Val loss:   0.0073

📊 Classification Report (Validation):
              precision    recall  f1-score   support

           0     0.9861    1.0000    0.9930       356
           1     1.0000    0.9993    0.9997      2988
           2     0.9991    0.9991    0.9991      3263
           3     1.0000    1.0000    1.0000        92
           4     1.0000    0.9975    0.9987      1191
           5     1.0000    1.0000    1.0000       831
           6     1.0000    1.0000    1.0000      1191
           7     1.0000    0.9973    0.9986      1830
           8     0.7500    1.0000    0.8571        30
           9     1.0000    1.0000    1.0000        31
          10     1.0000    1.0000    1.0000        16
          11     1.0000    0.9980    0.9990      2457
          12     1.0000    0.9983    0.9992      1816
          13     1.0000    0.9902    0.9951       713
          14     0.9622    0.9986    0.9800       713
          15     1.0000    1.0000    1.0000         8
          16     1.00

In [9]:
# ── NEW: Before/After comparison on the 16 confused pairs ─────────────────────
# Shows exactly whether class weighting reduced the specific error rates
# that the confusion analysis identified

CONFUSED_PAIRS = [
    ("CWE188", "CWE190", 0.013514),
    ("CWE176", "CWE190", 0.010811),
    ("CWE121", "CWE123", 0.002158),
    ("CWE122", "CWE123", 0.001660),
    ("CWE114", "CWE127", 0.009009),
    ("CWE121", "CWE190", 0.000719),
    ("CWE122", "CWE190", 0.000553),
    ("CWE123", "CWE15",  0.001252),
    ("CWE124", "CWE190", 0.002105),
    ("CWE126", "CWE15",  0.001252),
    ("CWE134", "CWE15",  0.000644),
    ("CWE134", "CWE190", 0.000644),
    ("CWE15",  "CWE190", 0.001152),
    ("CWE176", "CWE15",  0.002703),
    ("CWE188", "CWE15",  0.002703),
    ("CWE244", "CWE15",  0.004386),
]

# Map cwe_index → cwe name using val_df
idx_to_cwe  = dict(zip(mapping_df["cwe_index"], mapping_df["cwe"]))
preds_cwe   = [idx_to_cwe.get(p, str(p)) for p in y_pred]
labels_cwe  = [idx_to_cwe.get(l, str(l)) for l in y_true]
preds_s     = pd.Series(preds_cwe)
labels_s    = pd.Series(labels_cwe)

rows = []
print(f"{'Pair':<25} {'Before':>10} {'After':>10} {'Change':>10}")
print("-" * 58)

for true_cwe, pred_cwe, before_rate in CONFUSED_PAIRS:
    total    = (labels_s == true_cwe).sum()
    confused = ((labels_s == true_cwe) & (preds_s == pred_cwe)).sum()
    after_rate = confused / total if total > 0 else 0.0
    change     = after_rate - before_rate
    arrow      = "✅" if change < 0 else ("⚠️ " if change == 0 else "❌")

    print(f"{true_cwe}→{pred_cwe:<18} "
          f"{before_rate*100:>9.4f}%  "
          f"{after_rate*100:>9.4f}%  "
          f"{change*100:>+9.4f}%  {arrow}")

    rows.append({
        "True_CWE":       true_cwe,
        "Predicted_CWE":  pred_cwe,
        "Before_Rate":    round(before_rate, 6),
        "After_Rate":     round(after_rate,  6),
        "Change":         round(change,      6),
        "Improved":       change < 0,
    })

result_df = pd.DataFrame(rows)
improved  = result_df["Improved"].sum()
print(f"\nImproved: {improved}/{len(rows)} pairs")

result_df.to_csv("weighted_vs_original_confused_pairs.csv", index=False)
print("💾 Saved: weighted_vs_original_confused_pairs.csv")

Pair                          Before      After     Change
----------------------------------------------------------
CWE188→CWE190                1.3514%     0.0000%    -1.3514%  ✅
CWE176→CWE190                1.0811%     0.0000%    -1.0811%  ✅
CWE121→CWE123                0.2158%     0.0000%    -0.2158%  ✅
CWE122→CWE123                0.1660%     0.0000%    -0.1660%  ✅
CWE114→CWE127                0.9009%     0.0000%    -0.9009%  ✅
CWE121→CWE190                0.0719%     0.0000%    -0.0719%  ✅
CWE122→CWE190                0.0553%     0.0000%    -0.0553%  ✅
CWE123→CWE15                 0.1252%     0.0000%    -0.1252%  ✅
CWE124→CWE190                0.2105%     0.0000%    -0.2105%  ✅
CWE126→CWE15                 0.1252%     0.0000%    -0.1252%  ✅
CWE134→CWE15                 0.0644%     0.1639%    +0.0995%  ❌
CWE134→CWE190                0.0644%     0.0000%    -0.0644%  ✅
CWE15→CWE190                0.1152%     0.0000%    -0.1152%  ✅
CWE176→CWE15                 0.2703%     0.0000%   

In [10]:
# ── Save full classification report as CSV (same as original notebook) ─────────
report = classification_report(
    y_true, y_pred, digits=4, output_dict=True, zero_division=0
)
report_df = pd.DataFrame(report).transpose().reset_index()
report_df.rename(columns={"index": "cwe_index"}, inplace=True)
report_df = report_df[report_df["cwe_index"].str.isnumeric()]
report_df["cwe_index"] = report_df["cwe_index"].astype(int)
report_df = report_df.merge(mapping_df, on="cwe_index", how="left")

train_counts = train_df["cwe_index"].value_counts().to_dict()
val_counts   = val_df["cwe_index"].value_counts().to_dict()
report_df["train_samples"]      = report_df["cwe_index"].map(train_counts)
report_df["validation_samples"] = report_df["cwe_index"].map(val_counts)

final_df = report_df[[
    "cwe", "train_samples", "validation_samples",
    "precision", "recall", "f1-score", "support"
]].rename(columns={"f1-score": "f1"})

final_df.to_csv("cwe_metrics_summary_weighted.csv", index=False)
print("✅ Saved cwe_metrics_summary_weighted.csv")

✅ Saved cwe_metrics_summary_weighted.csv
